# 02 · Performance and risk metrics

**Goal of this notebook:** summarise 1,759 daily prices per company into a small set of performance and risk metrics, and join them with the ESG data.

**Inputs:** `data/prices_raw.csv`, `data/esg_clean.csv`
**Output:** `data/analysis_dataset.csv`

## 1. Load the clean data

In [1]:
import pandas as pd
import numpy as np

prices = pd.read_csv("../data/prices_raw.csv", index_col=0, parse_dates=True)
esg = pd.read_csv("../data/esg_clean.csv")

print(prices.shape, esg.shape)

(1759, 411) (411, 13)


## 2. Daily returns
Daily percentage change in price, the basis for most metrics. The first row is empty because there is no previous day to compare with.

In [2]:
# Daily percentage change in price
daily_returns = prices.pct_change()
daily_returns.head()

,A,AAL,AAPL,ABBV,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WRB,WTW,WY,WYNN,XEL,XOM,XYL,YUM,ZBH,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2019-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-01-03,-0.036840,-0.074507,-0.099607,-0.032949,-0.047194,-0.015655,-0.034142,-0.039498,-0.060405,-0.004395,...,-0.016570,-0.020978,0.033906,-0.009674,-0.003940,-0.015354,-0.034281,-0.025153,-0.018087,-0.034564
2019-01-04,0.034614,0.065868,0.042689,0.032217,0.028541,0.023662,0.038883,0.048632,0.024279,0.023056,...,0.015445,0.027024,0.034142,0.040333,0.009785,0.036870,0.042192,0.026027,0.034153,0.039235
2019-01-07,0.021234,0.028402,-0.002226,0.014595,0.014976,-0.002274,0.003474,0.013573,0.006288,0.005035,...,0.002904,0.007375,0.019983,0.017758,-0.004330,0.005200,-0.014789,-0.001093,0.000385,0.006017
2019-01-08,0.014660,-0.016085,0.019063,0.004648,-0.012006,0.003798,0.025290,0.014918,0.024396,0.018607,...,-0.007307,0.006398,0.005963,0.031333,0.011596,0.007271,0.009401,-0.001970,-0.022425,0.014894


## 3. Compute the metrics
For each company, five metrics are computed over 2019–2025:
- **Annual return (CAGR):** compound yearly return over the whole period.
- **Annualised volatility:** standard deviation of daily returns × √252 trading days.
- **Maximum drawdown:** worst fall from a previous peak.
- **COVID crash return:** return from the market peak (19 Feb 2020) to the bottom (23 Mar 2020).
- **2022 return:** return in the rate-hike year.

The last two act as **stress tests** to assess whether ESG scores relate to resilience during market downturns.

In [3]:
# Annualised return (CAGR) over the whole period
years = (prices.index[-1] - prices.index[0]).days / 365.25
annual_return = (prices.iloc[-1] / prices.iloc[0]) ** (1 / years) - 1

# Annualised volatility (252 trading days per year)
volatility = daily_returns.std() * np.sqrt(252)

# Maximum drawdown: worst fall from a previous peak
drawdown = prices / prices.cummax() - 1
max_drawdown = drawdown.min()

# Return during the COVID crash (market peak to bottom)
covid_crash = prices.loc["2020-03-23"] / prices.loc["2020-02-19"] - 1

# Return in 2022 (rate-hike year)
year_end = prices.resample("YE").last()
return_2022 = year_end.loc["2022"].iloc[0] / year_end.loc["2021"].iloc[0] - 1

## 4. Sanity check
The values are consistent with the market over the period: median annual return of 12%, median volatility of 32% (normal for single stocks) and a median COVID crash of −38%.

In [4]:
metrics = pd.DataFrame({
    "annual_return": annual_return,
    "volatility": volatility,
    "max_drawdown": max_drawdown,
    "covid_crash": covid_crash,
    "return_2022": return_2022
})
metrics.index.name = "ticker"
metrics = metrics.reset_index()

metrics.describe().round(3)

,annual_return,volatility,max_drawdown,covid_crash,return_2022
count,411.000,411.000,411.000,411.000,411.000
mean,0.128,0.333,-0.512,-0.380,-0.077
std,0.108,0.085,0.140,0.143,0.267
min,-0.176,0.192,-0.901,-0.849,-0.650
25%,0.064,0.274,-0.591,-0.459,-0.250
50%,0.120,0.317,-0.488,-0.380,-0.108
75%,0.186,0.370,-0.416,-0.301,0.065
max,0.776,0.701,-0.238,0.137,1.191


## 5. Join metrics with ESG data
Inner join on `ticker`: all 411 companies are kept, confirming that tickers match between both tables.

In [5]:
df = esg.merge(metrics, on="ticker", how="inner")
print(f"Companies in final dataset: {len(df)}")

df.to_csv("../data/analysis_dataset.csv", index=False)

Companies in final dataset: 411


## 6. Inspect extreme values
Extreme values are checked to make sure they are real and not data errors.

In [6]:
# Inspect extreme values
cols = ["ticker", "company_name", "sector"]
print(df.nlargest(3, "covid_crash")[cols + ["covid_crash"]])
print(df.nlargest(3, "return_2022")[cols + ["return_2022"]])
print(df.nlargest(3, "annual_return")[cols + ["annual_return"]])

    ticker                   company_name             sector  covid_crash
102   REGN  Regeneron Pharmaceuticals Inc         Healthcare     0.136773
1      DPZ            Domino's Pizza Inc.  Consumer Cyclical     0.092666
241   GILD            Gilead Sciences Inc         Healthcare     0.088139
    ticker                      company_name  sector  return_2022
139    OXY  Occidental Petroleum Corporation  Energy     1.190984
264    XOM           Exxon Mobil Corporation  Energy     0.874091
184    MPC    Marathon Petroleum Corporation  Energy     0.866228
    ticker   company_name             sector  annual_return
143   NVDA    Nvidia Corp         Technology       0.776186
64    TSLA    Tesla, Inc.  Consumer Cyclical       0.555696
348   AVGO  Broadcom Inc.         Technology       0.495206


The extreme values are real market outcomes, not errors, so they are kept:
- **COVID crash:** healthcare companies (Regeneron, Gilead) and Domino's rose during the crash.
- **2022:** oil companies (Occidental, Exxon Mobil, Marathon Petroleum) benefited from the oil price surge. These are also companies with high ESG risk, which matters for the analysis.
- **Annual return:** NVIDIA, Tesla and Broadcom led the period.

Next notebook: `03_eda.ipynb`, where the relationship between ESG risk and these metrics is analysed and tested.